<a href="https://colab.research.google.com/github/Innovatewithapple/Resume_JD_AnalysisTransfomer/blob/main/ResumeDecoderLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this cell only once when start working on project with new session. this cell is just for error fixing of torchao
!pip uninstall -y torchao
import os
os.kill(os.getpid(), 9)

In [ ]:
!pip show torchao

In [ ]:
!pip install -q transformers datasets peft accelerate trl bitsandbytes

In [ ]:
import torch
import random
import numpy as np
import os

def setupSeed(seed=32):
  os.environ['PYTHONHASHSEED'] = str(seed)
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True

setupSeed()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [36]:
from datasets import load_dataset
from peft import LoraConfig,get_peft_model
from datasets import Dataset as HFDataset
import pandas as pd
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from sklearn.model_selection import train_test_split
from peft import LoraConfig,get_peft_model
import trl
from trl import SFTConfig,SFTTrainer

In [ ]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16)

In [ ]:
#-------Tokenizer and Model----------!
Tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct',trust_remote_code=True)
Tokenizer.pad_token = Tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct',quantization_config=bnb_config,device_map='auto')

In [ ]:
ds = load_dataset("cnamuangtoun/resume-job-description-fit")

In [ ]:
resume_texts = ds['train']['resume_text']
jd_texts = ds['train']['job_description_text']
label_text = ds['train']['label']

val_resume_texts = ds['test']['resume_text']
val_jd_texts = ds['test']['job_description_text']
val_label_text = ds['test']['label']

In [ ]:
label_counts = pd.Series(label_text).value_counts()

print(label_counts)

print("\nPercentages:")
print((label_counts / len(label_text) * 100).round(2))

In [ ]:
#------Build Dataset------!
train_Ds = HFDataset.from_dict({'resume':resume_texts,'jd':jd_texts,'label':label_text})
val_Ds = HFDataset.from_dict({'resume':val_resume_texts,'jd':val_jd_texts,'label':val_label_text})

In [ ]:
def message_Format(data):
  messages = [
      {
          'role':"system",'content':('You are an expert recruiter and talent acquisition specialist.''Evaluate how well a candidate matches a job description and''classify the candidate into one of the provided categories.')
      },

      {
          'role':'user','content':f"""Resume: {data['resume']} Job Description: {data['jd']}
          Classify the candidate as one of :
           Good Fit
           Potential Fit
           No Fit
          """
      },
      {
          'role':'assistant','content':data['label']
      }
  ]
  return {"messages": messages}

In [ ]:
train_Ds = train_Ds.map(message_Format)
val_Ds = val_Ds.map(message_Format)

In [ ]:
#-------Apply Chat Template-------!
def apply_template(prompt):
  text = Tokenizer.apply_chat_template(prompt['messages'],tokenize=False)
  return {'text':text}

In [ ]:
train_Ds = train_Ds.map(apply_template)
val_Ds = val_Ds.map(apply_template)

In [ ]:
#---------Check module names for Qlora---------!
for name, module in model.named_modules():
  if 'proj' in name:
    print(name)

In [ ]:
#--------Lora Configuration-------!
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout= 0.05,
    bias= 'none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
)

In [ ]:
#-------Attach Lora-----!
model = get_peft_model(model,lora_config)
model.print_trainable_parameters()

In [42]:
sft_config = SFTConfig(
    output_dir='./Qwen_resumeMatcher',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10, #print training steps
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    max_length=3072,
    fp16=True,
    report_to='none'
)

model.config.use_cache = False
trainer = SFTTrainer(
    model=model,
    train_dataset=train_Ds,
    eval_dataset=val_Ds,
    args=sft_config,
    processing_class=Tokenizer
)

Tokenizing train dataset:   0%|          | 0/6241 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1759 [00:00<?, ? examples/s]

In [ ]:
trainer.train()
